In [64]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [65]:
df = pd.read_csv("train_clean(1).csv")

In [66]:
df.head

<bound method NDFrame.head of        Patient Age Genes in mother's side Inherited from father Maternal gene  \
0              2.0                    Yes                    No           Yes   
1              4.0                    Yes                   Yes            No   
2              6.0                    Yes                    No            No   
3             12.0                    Yes                    No           Yes   
4             11.0                    Yes                    No           Yes   
...            ...                    ...                   ...           ...   
22078          4.0                    Yes                   Yes           Yes   
22079          8.0                     No                   Yes            No   
22080          8.0                    Yes                    No           Yes   
22081          7.0                    Yes                    No           Yes   
22082         11.0                    Yes                    No            No  

In [67]:
df.head()

,Patient Age,Genes in mother's side,Inherited from father,Maternal gene,Paternal gene,Blood cell count (mcL),Mother's age,Father's age,Status,Respiratory Rate (breaths/min),...,Symptom 5,missing_count,symptomCount,symptomMean,symptomMax,symptomStd,symptomMin,symptomRange,Genetic Disorder,Disorder Subclass
0,2.0,Yes,No,Yes,No,4.760603,35.0,42.0,Alive,Normal (30-60),...,1.0,0,5.0,1.0,1.0,0.000000,1.0,0.0,Mitochondrial genetic inheritance disorders,Leber's hereditary optic neuropathy
1,4.0,Yes,Yes,No,No,4.910669,35.0,23.0,Deceased,Tachypnea,...,0.0,0,3.0,0.6,1.0,0.547723,0.0,1.0,Mitochondrial genetic inheritance disorders,Cystic fibrosis
2,6.0,Yes,No,No,No,4.893297,41.0,22.0,Alive,Normal (30-60),...,1.0,0,4.0,0.8,1.0,0.447214,0.0,1.0,Multifactorial genetic inheritance disorders,Diabetes
3,12.0,Yes,No,Yes,No,4.705280,21.0,42.0,Deceased,Tachypnea,...,0.0,0,1.0,0.2,1.0,0.447214,0.0,1.0,Mitochondrial genetic inheritance disorders,Leigh syndrome
4,11.0,Yes,No,Yes,Yes,4.720703,32.0,42.0,Alive,Tachypnea,...,0.0,0,0.0,0.0,0.0,0.000000,0.0,0.0,Multifactorial genetic inheritance disorders,Other


In [68]:
df = df.drop(columns=["Mother's age","Father's age"])

In [69]:
df.shape

(22083, 41)

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

%matplotlib inline
sns.set_style("whitegrid")

In [71]:
display(df.describe())

,Patient Age,Blood cell count (mcL),Test 1,Test 2,Test 3,Test 4,Test 5,No. of previous abortion,White Blood cell count (thousand per microliter),Symptom 1,...,Symptom 3,Symptom 4,Symptom 5,missing_count,symptomCount,symptomMean,symptomMax,symptomStd,symptomMin,symptomRange
count,22083.000000,22083.000000,22083.0,22083.0,22083.0,22083.0,22083.0,22083.000000,22083.000000,22083.000000,...,22083.000000,22083.000000,22083.000000,22083.0,22083.000000,22083.000000,22083.000000,22083.000000,22083.000000,22083.000000
mean,6.975819,4.898871,0.0,0.0,0.0,1.0,0.0,2.002762,7.485340,0.534665,...,0.485215,0.450120,0.416882,0.0,2.383236,0.476647,0.964271,0.477054,0.033057,0.931214
std,4.177581,0.199663,0.0,0.0,0.0,0.0,0.0,1.341020,2.521040,0.498808,...,0.499793,0.497517,0.493054,0.0,1.142657,0.228531,0.185617,0.137690,0.178790,0.253096
min,0.000000,4.092727,0.0,0.0,0.0,1.0,0.0,0.000000,3.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,4.763109,0.0,0.0,0.0,1.0,0.0,1.000000,5.653911,0.000000,...,0.000000,0.000000,0.000000,0.0,2.000000,0.400000,1.000000,0.447214,0.000000,1.000000
50%,7.000000,4.899399,0.0,0.0,0.0,1.0,0.0,2.000000,7.477132,1.000000,...,0.000000,0.000000,0.000000,0.0,2.000000,0.400000,1.000000,0.547723,0.000000,1.000000
75%,10.000000,5.033830,0.0,0.0,0.0,1.0,0.0,3.000000,9.279380,1.000000,...,1.000000,1.000000,1.000000,0.0,3.000000,0.600000,1.000000,0.547723,0.000000,1.000000
max,14.000000,5.609829,0.0,0.0,0.0,1.0,0.0,4.000000,12.000000,1.000000,...,1.000000,1.000000,1.000000,0.0,5.000000,1.000000,1.000000,0.547723,1.000000,1.000000


In [72]:
df['Genetic Disorder'].unique()

array(['Mitochondrial genetic inheritance disorders',
       'Multifactorial genetic inheritance disorders',
       'Single-gene inheritance diseases'], dtype=object)

In [73]:
X = df.drop(['Genetic Disorder', 'Disorder Subclass'], axis=1).copy()
X = pd.get_dummies(X, dummy_na=True)

In [74]:
X

,Patient Age,Blood cell count (mcL),Test 1,Test 2,Test 3,Test 4,Test 5,No. of previous abortion,White Blood cell count (thousand per microliter),Symptom 1,...,History of anomalies in previous pregnancies_Yes,History of anomalies in previous pregnancies_nan,Birth defects_Multiple,Birth defects_Singular,Birth defects_nan,Blood test result_abnormal,Blood test result_inconclusive,Blood test result_normal,Blood test result_slightly abnormal,Blood test result_nan
0,2.0,4.760603,0.0,0.0,0.0,1.0,0.0,2.0,9.857562,1.0,...,True,False,False,True,False,False,False,False,True,False
1,4.0,4.910669,0.0,0.0,0.0,1.0,0.0,2.0,5.522560,1.0,...,True,False,True,False,False,False,False,True,False,False
2,6.0,4.893297,0.0,0.0,0.0,1.0,0.0,4.0,7.477132,0.0,...,True,False,False,True,False,False,False,True,False,False
3,12.0,4.705280,0.0,0.0,0.0,1.0,0.0,1.0,7.919321,0.0,...,True,False,False,True,False,False,True,False,False,False
4,11.0,4.720703,0.0,0.0,0.0,1.0,0.0,4.0,4.098210,0.0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22078,4.0,5.258298,0.0,0.0,0.0,1.0,0.0,3.0,6.584811,0.0,...,False,False,True,False,False,False,True,False,False,False
22079,8.0,4.974220,0.0,0.0,0.0,1.0,0.0,2.0,7.041556,1.0,...,False,False,True,False,False,False,True,False,False,False
22080,8.0,5.186470,0.0,0.0,0.0,1.0,0.0,2.0,7.715464,0.0,...,False,False,False,True,False,False,False,True,False,False
22081,7.0,4.858543,0.0,0.0,0.0,1.0,0.0,1.0,8.437670,1.0,...,False,False,True,False,False,True,False,False,False,False


In [75]:
mapping_dict = {
    'Mitochondrial genetic inheritance disorders': 0,
    'Multifactorial genetic inheritance disorders': 1,
    'Single-gene inheritance diseases': 2
}

y = df['Genetic Disorder'].map(mapping_dict)

In [76]:
y

0        0
1        0
2        1
3        0
4        1
        ..
22078    0
22079    1
22080    0
22081    0
22082    1
Name: Genetic Disorder, Length: 22083, dtype: int64

In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [78]:
params = {
    'objective': 'multi:softmax',  
    'num_class': 3,                
    'max_depth': 4,                
    'learning_rate': 0.1,          
    'n_estimators': 100,           
    'alpha': 10,              
    'eval_metric': 'mlogloss'
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy for Genetic Disorder:", accuracy)
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Model Accuracy for Genetic Disorder: 0.6076981132075472

Classification Report
              precision    recall  f1-score   support

           0       0.65      0.84      0.74      3729
           1       0.45      0.26      0.33       654
           2       0.50      0.32      0.39      2242

    accuracy                           0.61      6625
   macro avg       0.53      0.47      0.49      6625
weighted avg       0.58      0.61      0.58      6625



In [79]:
X

,Patient Age,Blood cell count (mcL),Test 1,Test 2,Test 3,Test 4,Test 5,No. of previous abortion,White Blood cell count (thousand per microliter),Symptom 1,...,History of anomalies in previous pregnancies_Yes,History of anomalies in previous pregnancies_nan,Birth defects_Multiple,Birth defects_Singular,Birth defects_nan,Blood test result_abnormal,Blood test result_inconclusive,Blood test result_normal,Blood test result_slightly abnormal,Blood test result_nan
0,2.0,4.760603,0.0,0.0,0.0,1.0,0.0,2.0,9.857562,1.0,...,True,False,False,True,False,False,False,False,True,False
1,4.0,4.910669,0.0,0.0,0.0,1.0,0.0,2.0,5.522560,1.0,...,True,False,True,False,False,False,False,True,False,False
2,6.0,4.893297,0.0,0.0,0.0,1.0,0.0,4.0,7.477132,0.0,...,True,False,False,True,False,False,False,True,False,False
3,12.0,4.705280,0.0,0.0,0.0,1.0,0.0,1.0,7.919321,0.0,...,True,False,False,True,False,False,True,False,False,False
4,11.0,4.720703,0.0,0.0,0.0,1.0,0.0,4.0,4.098210,0.0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22078,4.0,5.258298,0.0,0.0,0.0,1.0,0.0,3.0,6.584811,0.0,...,False,False,True,False,False,False,True,False,False,False
22079,8.0,4.974220,0.0,0.0,0.0,1.0,0.0,2.0,7.041556,1.0,...,False,False,True,False,False,False,True,False,False,False
22080,8.0,5.186470,0.0,0.0,0.0,1.0,0.0,2.0,7.715464,0.0,...,False,False,False,True,False,False,False,True,False,False
22081,7.0,4.858543,0.0,0.0,0.0,1.0,0.0,1.0,8.437670,1.0,...,False,False,True,False,False,True,False,False,False,False


In [80]:
df['Disorder Subclass'].unique()

array(["Leber's hereditary optic neuropathy", 'Cystic fibrosis',
       'Diabetes', 'Leigh syndrome', 'Other', 'Tay-Sachs',
       'Hemochromatosis', 'Mitochondrial myopathy'], dtype=object)

In [81]:
mapping_dict = {
    'Leigh syndrome': 0,
    'Diabetes': 1,
    'Mitochondrial myopathy': 0,
    'Hemochromatosis': 2,
    'Cystic fibrosis': 2,
    'Tay-Sachs': 2,
    'Alzheimer s': 1,
    'Leber s hereditary optic neuropathy': 0,
    'Cancer': 1
}

y = df['Disorder Subclass'].map(mapping_dict)


valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx].astype(int)

In [82]:
y

1        2
2        1
3        0
5        2
6        2
        ..
22078    0
22079    1
22080    0
22081    0
22082    1
Name: Disorder Subclass, Length: 21186, dtype: int64

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [84]:
params = {
    'objective': 'multi:softmax',  
    'num_class': 3,                
    'max_depth': 4,                
    'learning_rate': 0.1,          
    'n_estimators': 100,           
    'alpha': 10,              
    'eval_metric': 'mlogloss'
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy for Disorder Subclass:", accuracy)
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Model Accuracy for Disorder Subclass: 0.6235053492762744

Classification Report
              precision    recall  f1-score   support

           0       0.65      0.87      0.74      3473
           1       0.53      0.31      0.39       559
           2       0.56      0.33      0.42      2324

    accuracy                           0.62      6356
   macro avg       0.58      0.50      0.52      6356
weighted avg       0.61      0.62      0.59      6356

